# Fast c-GC simulation rerun - FastICA split

Runs only `cgc` and `cgc_star` with the fast c-GC backend for the FastICA split. This writes to `outputs/simulation/core_1000_cgc_fast_fastica/` and can run in parallel with the other fast c-GC split notebooks.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "ica_denoising").exists():
            return path
    raise RuntimeError("Could not find the ica-denoising repository root.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".cache" / "matplotlib"))
(REPO_ROOT / ".cache" / "matplotlib").mkdir(parents=True, exist_ok=True)

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from ica_denoising.simulation import load_config, run_benchmark

CONFIG_PATH = REPO_ROOT / "configs" / "simulation.core.cgc_fast.fastica.json"
cfg = load_config(CONFIG_PATH)
benchmark_root = REPO_ROOT / cfg.run.output_dir / cfg.benchmark_version

print(json.dumps({
    "config": str(CONFIG_PATH.relative_to(REPO_ROOT)),
    "benchmark_root": str(benchmark_root.relative_to(REPO_ROOT)),
    "estimators": list(cfg.estimator.estimators),
    "cgc_backend": cfg.estimator.cgc_backend,
    "n_perm": cfg.estimator.n_perm,
    "correction": cfg.estimator.correction,
    "beta": cfg.estimator.beta,
}, indent=2))

In [ ]:
def progress(event: dict) -> None:
    if event.get("event") in {"done", "skipped"}:
        print(
            f"{event['event']}: {event['scenario_id']} seed {event['seed']} "
            f"({event['completed']}/{event['total']})"
        )


paths = run_benchmark(cfg, resume=True, progress_callback=progress)
print(f"Finished {len(paths)} replicate(s).")
print(f"Output root: {benchmark_root.relative_to(REPO_ROOT)}")